# Phase II_02: U-Net Training on Change Detection

**Objective**: Train U-Net to classify burn severity from spectral change (Post - Pre)

**Key innovation**: Difference-based input allows direct Phase III transfer to NAIP

**Input**: 4-channel change images (RGBN difference)

**Output**: 7-class burn severity labels

## Setup: Mount Drive and Load Dependencies

In [ ]:
%pip install -q torch torchvision numpy matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from datetime import datetime

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✓ Setup complete")
print("Note: This notebook loads pre-computed difference images from Phase II_01")

In [ ]:
class ChangeDetectionDataset(Dataset):
    """Dataset that loads pre-computed (Post - Pre) difference images from Phase II_01.
    
    Phase II_01 now saves difference_images_rgbn_*.pt containing:
    - 424 samples × [4 channels (RGBN), 512×512 resolution]
    - Difference = (Post - Pre) for each channel
    """
    
    def __init__(self, diff_images_tensor, labels_tensor, split='train', fold_path=None):
        """
        Args:
            diff_images_tensor: [N, 4, 512, 512] pre-computed differences from Phase II_01
            labels_tensor: [N, 512, 512] severity labels from Phase II_01
            split: 'train', 'val', or 'test'
            fold_path: Optional path to metadata JSON for fold information
        """
        self.diff_images = diff_images_tensor
        self.labels = labels_tensor
        self.split = split
        
        # If fold information available, filter by split
        if fold_path and Path(fold_path).exists():
            with open(fold_path, 'r') as f:
                metadata = json.load(f)
            
            # Build fold mapping from metadata
            self.fold_map = {}
            sample_idx = 0
            for fire in metadata['fires']:
                for sample in fire['samples']:
                    fold_name = sample['fold']
                    self.fold_map[sample_idx] = fold_name
                    sample_idx += 1
            
            # Filter indices by split
            fold_to_split = {'val': 'val', 'train': 'train', 'test': 'test'}
            self.indices = [i for i in range(len(self.diff_images)) 
                           if fold_to_split.get(self.fold_map.get(i), 'unknown') == split]
        else:
            # If no fold info, use all samples (warning: this mixes train/val/test)
            self.indices = list(range(len(self.diff_images)))
            print(f"⚠ Warning: No fold information provided. Using all {len(self.indices)} samples for split '{split}'")
        
        print(f"✓ Dataset '{split}': {len(self.indices)} samples from {len(self.diff_images)} total")
    
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        sample_idx = self.indices[idx]
        
        # Get pre-computed difference image and label
        diff_image = self.diff_images[sample_idx].float()  # [4, 512, 512]
        label = self.labels[sample_idx].long()  # [512, 512]
        
        return diff_image, label

# Load Phase II_01 outputs from Drive
print(f"Loading Phase II_01 outputs from Drive...\n")

labels_dir = Path('/content/drive/MyDrive/RETINNA_DATA/phase2/II_01_relabeling')
if not labels_dir.exists():
    raise FileNotFoundError(f"Phase II_01 outputs not found at {labels_dir}")

# Find latest files
label_files = sorted(labels_dir.glob('multi_class_labels_*.pt'))
diff_files = sorted(labels_dir.glob('difference_images_rgbn_*.pt'))

if not label_files or not diff_files:
    raise FileNotFoundError(f"Missing Phase II_01 outputs in {labels_dir}")

latest_labels_file = label_files[-1]
latest_diffs_file = diff_files[-1]

print(f"✓ Loading labels: {latest_labels_file.name}")
labels_tensor = torch.load(latest_labels_file)
print(f"  Shape: {labels_tensor.shape}")

print(f"✓ Loading difference images: {latest_diffs_file.name}")
diffs_tensor = torch.load(latest_diffs_file)
print(f"  Shape: {diffs_tensor.shape}")

# Try to load metadata for fold information (optional)
metadata_paths = [
    Path('/content/drive/MyDrive/RETINNA_DATA/metadata/cabuaur_metadata.json'),
    Path('/content/RETINNA/docs/cabuaur_metadata.json'),
]
metadata_path = None
for path in metadata_paths:
    if path.exists():
        metadata_path = path
        print(f"\n✓ Found metadata at {path}")
        break

if not metadata_path:
    print(f"\n⚠ Metadata not found. Will use all {len(diffs_tensor)} samples (train/val mixed)")

# Create datasets with fold information
train_dataset = ChangeDetectionDataset(diffs_tensor, labels_tensor, split='train', fold_path=metadata_path)
val_dataset = ChangeDetectionDataset(diffs_tensor, labels_tensor, split='val', fold_path=metadata_path)

print(f"\n✓ Datasets loaded successfully")

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    
    def forward(self, x):
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels // 2, out_channels)
    
    def forward(self, x1, x2):
        x1 = self.up(x1)
        # Handle size mismatch from skip connection
        if x2.size() != x1.size():
            x1 = F.pad(x1, (0, x2.size(3) - x1.size(3), 0, x2.size(2) - x1.size(2)))
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    """U-Net for 7-class burn severity segmentation from 4-channel difference images."""
    def __init__(self, in_channels=4, out_channels=7, bilinear=True):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.bilinear = bilinear
        
        factor = 2 if bilinear else 1
        
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024 // factor)
        
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        
        x = self.outc(x)
        return x

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = UNet(in_channels=4, out_channels=7, bilinear=True).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model: U-Net")
print(f"  Input channels: 4 (RGBN difference)")
print(f"  Output classes: 7 (burn severity)")
print(f"  Total parameters: {total_params:,}")
print(f"  Device: {device}")

## Define U-Net Architecture

In [ ]:
# Training configuration
batch_size = 4  # Adjust based on GPU memory
learning_rate = 1e-3
num_epochs = 20
weight_decay = 1e-5

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Loss function with class weights (to handle imbalance: 95% Low vs 1% Extreme)
# Class weights: inverse of frequency
class_weights = torch.tensor([1.0, 0.01, 1.0, 1.0, 1.0, 1.0, 1.0]).to(device)  # Adjust based on actual distribution
criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"Training configuration:")
print(f"  Batch size: {batch_size}")
print(f"  Learning rate: {learning_rate}")
print(f"  Epochs: {num_epochs}")
print(f"  Optimizer: Adam")
print(f"  Loss: Weighted CrossEntropyLoss")
print(f"\nData loaders:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

## Training Loop

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_pixels = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Metrics
        total_loss += loss.item()
        predictions = outputs.argmax(dim=1)
        total_correct += (predictions == labels).sum().item()
        total_pixels += labels.numel()
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx + 1}/{len(train_loader)}, Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / len(train_loader)
    accuracy = total_correct / total_pixels
    return avg_loss, accuracy

def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_pixels = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            predictions = outputs.argmax(dim=1)
            total_correct += (predictions == labels).sum().item()
            total_pixels += labels.numel()
    
    avg_loss = total_loss / len(val_loader)
    accuracy = total_correct / total_pixels
    return avg_loss, accuracy

# Training loop
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

## Visualization and Metrics

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid()

axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'], label='Val')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid()

plt.tight_layout()
plt.savefig('training_history.png', dpi=100)
plt.show()

print(f"✓ Training history saved")

## Save Model

In [ ]:
from datetime import datetime

# Save model
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_save_path = f'/content/drive/MyDrive/RETINNA_cache/phase2/II_02_training/unet_model_{timestamp}.pt'

# Create directory if it doesn't exist
model_dir = Path(model_save_path).parent
model_dir.mkdir(parents=True, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'epoch': num_epochs,
    'config': {
        'in_channels': 4,
        'out_channels': 7,
        'input_type': 'RGBN difference (Post - Pre)',
        'label_source': 'Phase II_01 RdNBR classification',
    },
    'history': history,
}, model_save_path)

print(f"✓ Model saved: {model_save_path}")
print(f"  Final train loss: {history['train_loss'][-1]:.4f}")
print(f"  Final val loss: {history['val_loss'][-1]:.4f}")
print(f"  Final train acc: {history['train_acc'][-1]:.4f}")
print(f"  Final val acc: {history['val_acc'][-1]:.4f}")

## Summary

✓ **Phase II_02 Complete**

**Model trained on:**
- Input: 4-channel difference images (Post - Pre) for RGBN
- Output: 7-class burn severity segmentation
- Architecture: U-Net with change-detection design

**Ready for Phase III:**
- Model trained on NAIP-compatible bands
- Can directly transfer to NAIP temporal pairs
- No feature extraction or band adaptation needed

**Next steps:**
1. Acquire NAIP temporal pairs (pre/post fire)
2. Compute difference images
3. Run inference
4. Validate results